# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saif-Ullah0/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The FlyRank SEO Research paper (March 2026) presents
findings on content performance and search visibility.
I am applying the same audit lens Mirza used in the
Week 6 session to two specific findings.

FINDING 1: The paper reports a relationship between
content freshness and search ranking performance.

Methodology question: Where does the freshness signal
come from and is it measured at the right moment?
Specifically, is days_since_last_update recorded at the
time of the ranking observation, or at a later audit
date? If the timestamp is from a post-hoc audit rather
than the observation window, the freshness signal may
reflect editorial decisions made after the ranking
outcome was already visible, which would introduce
a form of look-ahead bias into any model trained on it.

This is a constructive question, not a critique. The
paper was built for a broad audience and the finding
is directionally plausible. The question is about
whether the validation design supports a causal claim
versus a correlational observation.

FINDING 2: The paper reports that pages with higher
impressions in the 90-day window tend to maintain
rankings better than lower-impression pages.

Methodology question: Is the impression signal
aggregated across the full 90-day window or weighted
toward recent days? A page that had high impressions
60 to 90 days ago but is currently declining would
show high impressions_90d while actually being in
decay. If the validation split does not account for
this temporal mismatch, the model may learn a signal
that is informative in aggregate but misleading at
the individual page level.

Again, this is not a flaw — it is the kind of question
that sharpens how findings should be stated.
"Pages with higher trailing impressions are
directionally more stable" is a safer claim than
"high impression pages maintain rankings."

In [2]:
import os, json
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/Saif-Ullah0/flyrank-ml-internship.git
os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df)} rows, declining rate: {df['is_declining_label'].mean():.3f}")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 165 (delta 69), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 1.87 MiB | 8.97 MiB/s, done.
Resolving deltas: 100% (69/69), done.
Loaded 30000 rows, declining rate: 0.542


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

features = [
    "impressions_90d", "ctr", "avg_position",
    "days_since_last_update", "content_age_days",
    "word_count", "engagement_rate", "scroll_rate",
    "ai_traffic_pct"
]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
groups = df["client_id"].values

# BEFORE: Random split (naive, dishonest)
from sklearn.model_selection import train_test_split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf_random = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    max_depth=10, random_state=42, n_jobs=-1
)
rf_random.fit(X_tr_r, y_tr_r)
score_random = precision_at_k(
    rf_random.predict_proba(X_te_r)[:, 1], y_te_r
)

# AFTER: Client-holdout split (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

rf_grouped = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    max_depth=10, random_state=42, n_jobs=-1
)
rf_grouped.fit(X_train, y_train)
score_grouped = precision_at_k(
    rf_grouped.predict_proba(X_test)[:, 1], y_test
)

print("SPLIT DESIGN COMPARISON")
print("=" * 45)
print(f"{'Split type':<30} {'Precision@50':>12}")
print("=" * 45)
print(f"{'Random split (naive)':<30} {score_random:.3f}")
print(f"{'Client-holdout (honest)':<30} {score_grouped:.3f}")
print("=" * 45)
print(f"\nDifference: {score_random - score_grouped:+.3f}")
if score_random > score_grouped:
    print("Random split inflates the score by leaking")
    print("client-specific patterns into the test set.")
    print("Client-holdout is the honest number.")

SPLIT DESIGN COMPARISON
Split type                     Precision@50
Random split (naive)           0.920
Client-holdout (honest)        0.640

Difference: +0.280
Random split inflates the score by leaking
client-specific patterns into the test set.
Client-holdout is the honest number.


The before/after comparison shows whether the random
split inflates performance by allowing the model to
learn client-specific patterns. If random split scores
higher than client-holdout, the difference is the
amount of optimism the naive split introduces.

My Week 5 model already used the client-holdout split.
This section confirms that choice was correct and
quantifies what the inflated score would have been.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
print("LEAKAGE AUDIT")
print("=" * 55)

# Features used in final model
used_features = features
print(f"Features in model ({len(used_features)}):")
for f in used_features:
    print(f"  {f}")

# Features explicitly excluded and why
excluded = {
    "trend_direction": "IS the label source. Direct leakage.",
    "trend_pct": "Derived from label. Leakage confirmed in Notebook 02.",
    "impressions_last_30d": "Encodes recent month signal used to compute trend_direction.",
    "impressions_prev_30d": "Encodes previous month signal used to compute trend_direction.",
    "clicks_last_30d": "Same temporal leakage as impressions_last_30d.",
    "clicks_prev_30d": "Same temporal leakage as impressions_prev_30d.",
}

print(f"\nExcluded features ({len(excluded)}) with leakage reason:")
for feat, reason in excluded.items():
    print(f"  {feat}: {reason}")

# Verify none of excluded features are in used list
leaky_in_model = [f for f in excluded if f in used_features]
if leaky_in_model:
    print(f"\nWARNING: Leaky features still in model: {leaky_in_model}")
else:
    print(f"\nLeakage check: CLEAN. No excluded features in model.")

# Check if any feature correlates suspiciously with label
print("\nCorrelation of features with label (top 5):")
corr = pd.DataFrame(X_train).copy()
corr['label'] = y_train
correlations = corr.corr()['label'].drop('label').abs().sort_values(ascending=False)
print(correlations.head(5).round(4).to_string())
print("\nNote: High correlation is not leakage by itself.")
print("Leakage requires the feature to encode future information.")

LEAKAGE AUDIT
Features in model (9):
  impressions_90d
  ctr
  avg_position
  days_since_last_update
  content_age_days
  word_count
  engagement_rate
  scroll_rate
  ai_traffic_pct

Excluded features (6) with leakage reason:
  trend_direction: IS the label source. Direct leakage.
  trend_pct: Derived from label. Leakage confirmed in Notebook 02.
  impressions_last_30d: Encodes recent month signal used to compute trend_direction.
  impressions_prev_30d: Encodes previous month signal used to compute trend_direction.
  clicks_last_30d: Same temporal leakage as impressions_last_30d.
  clicks_prev_30d: Same temporal leakage as impressions_prev_30d.

Leakage check: CLEAN. No excluded features in model.

Correlation of features with label (top 5):
content_age_days          0.1838
word_count                0.1306
days_since_last_update    0.1056
ctr                       0.0700
avg_position              0.0175

Note: High correlation is not leakage by itself.
Leakage requires the feature to e

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
# Claim rewrite summary
claims = {
    "ORIGINAL": "Random Forest outperforms the baseline.",
    "REWRITTEN": "Random Forest matched baseline Precision@50 = 0.640 on held-out client data. Directional result only.",
    "SPLIT_INFLATION": f"Random split would have shown {0.920:.3f} vs honest {0.640:.3f} (+0.280 optimism)",
    "LEAKAGE_REMOVED": "6 temporally leaky features removed before training.",
    "SAFE_CLAIM": "Observed Precision@50 = 0.640 under client-holdout. Decision-support tool only."
}

print("CLAIM AUDIT SUMMARY")
print("=" * 60)
for k, v in claims.items():
    print(f"{k}:")
    print(f"  {v}")
    print()

CLAIM AUDIT SUMMARY
ORIGINAL:
  Random Forest outperforms the baseline.

REWRITTEN:
  Random Forest matched baseline Precision@50 = 0.640 on held-out client data. Directional result only.

SPLIT_INFLATION:
  Random split would have shown 0.920 vs honest 0.640 (+0.280 optimism)

LEAKAGE_REMOVED:
  6 temporally leaky features removed before training.

SAFE_CLAIM:
  Observed Precision@50 = 0.640 under client-holdout. Decision-support tool only.



ORIGINAL CLAIMS (from Week 5 notebook) vs REWRITTEN
CLAIMS using safe public language.

---

ORIGINAL: "Random Forest outperforms the baseline."
PROBLEM: The model tied the baseline at 0.640. This
claim is factually wrong.
REWRITTEN: "Random Forest matched the baseline
Precision@50 of 0.640 on the 30,000-row sample under
a client-holdout split. Whether this result holds on
the full 79M row dataset is the open question for
Week 8."

---

ORIGINAL: "The model is better at finding declining pages."
PROBLEM: At Precision@50 = 0.640, 36% of the top 50
flagged pages are false positives. "Better" overstates
the result.
REWRITTEN: "The model identifies declining pages at
an observed rate of 0.640 Precision@50 on held-out
client data. This is directionally useful as a
decision-support tool for content teams, not a
definitive classifier."

---

ORIGINAL: "Impressions and position are the key signals."
PROBLEM: Permutation importance showed impressions_90d
had meaningful importance but word_count and
days_since_last_update had near-zero permutation
importance despite appearing important in tree-based
importance scores.
REWRITTEN: "Permutation importance on held-out data
suggests impressions_90d and CTR are the most
consistently predictive signals in this sample.
Content age and staleness showed limited permutation
importance, suggesting the model does not rely on
them as strongly as tree-based importance scores
imply."

---

ORIGINAL: "The leaky features were removed."
CORRECT as stated but incomplete.
REWRITTEN: "Six features were identified as temporally
leaky because they encode the month-over-month
impression comparison that generates the trend_direction
label. These were removed before training. The remaining
features represent signals knowable at the moment a
content intervention decision would be made."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.